# Logistic regression in PyTorch: MNIST and Imagenette

**COMP3620 Neural Networks and Deep Learning — Coursework 2, notebook 2.**
Understanding Deep Learning (UDL), Chapters 5 and 6.

Coursework 2 takes four steps (the worksheet's Parts 1–4):

1. **Logistic regression, the naive way** — notebook 1,
   `COMP3620_CW2_1_logistic_regression_numpy.ipynb`, which writes the softmax, the gradient and the update in NumPy,
   one pattern at a time. Do it first.
2. **The same model in PyTorch** — Part 2 of this notebook,
   `COMP3620_CW2_2_logistic_regression_pytorch.ipynb`.
3. **Realistic images** — Part 3 of this notebook. Logistic regression works
   on MNIST; does it work on realistic images as well? You find out on
   Imagenette.
4. **A small convolutional network** on Imagenette — notebook 3,
   `COMP3620_CW2_3_cnn_imagenette.ipynb`.

The worksheet says what to report. Cells marked **TODO** are yours to
complete; until you do, the runs that need them print a reminder and are
skipped, so the notebook always runs from top to bottom.

Run it on a GPU. Every run below takes seconds to about half a minute
there. The first time, Part 3 downloads Imagenette (1.5 GB) and decodes
the images once (a few minutes); later runs read a cached copy.

In [ ]:
import pathlib, time
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torchvision import datasets, transforms
from PIL import Image

device = ('cuda' if torch.cuda.is_available() else
          'mps' if torch.backends.mps.is_available() else 'cpu')
print('torch', torch.__version__, '| device:', device)

DATA_ROOT = pathlib.Path('./data')   # MNIST and Imagenette are downloaded here

## Part 2. Logistic regression on MNIST, in PyTorch

### 2.1 Data

The same data as notebook 1: the 60,000 MNIST training images, each
flattened to a row of 784 pixels in $[0, 1]$; the **first 80%** of the rows
for training and the **last 20%** for testing.

Two things are no longer needed. PyTorch's loss takes the **label** of each
image (an integer 0–9), not a one-hot target vector. And the bias is a
separate parameter of the linear layer, so there is no column of ones.

In [ ]:
mnist = datasets.MNIST(DATA_ROOT, train=True, download=True)
X = mnist.data.reshape(len(mnist), -1).float() / 255     # 60000 x 784, as ToTensor()
y = mnist.targets.clone()                                 # labels 0..9

n_train = int(0.8 * len(X))
MNIST_DATA = dict(name='MNIST', X_train=X[:n_train], y_train=y[:n_train],
                  X_test=X[n_train:], y_test=y[n_train:])
print(f"training: {len(MNIST_DATA['X_train'])} images   test: {len(MNIST_DATA['X_test'])} images"
      f"   pixels per image: {X.shape[1]}")

### 2.2 From NumPy to PyTorch

| Notebook 1 (NumPy) | PyTorch |
|---|---|
| `weights`, a $785 \times 10$ matrix (bias as the last row) | `nn.Linear(784, 10)`: a $10 \times 784$ weight matrix and 10 biases |
| `stableSoftmax` followed by the cross-entropy loss | `nn.CrossEntropyLoss()`, which takes the scores and applies a stable softmax itself |
| `gradient(...)`, derived and coded by hand | `loss.backward()`: PyTorch computes the gradient of the loss for every parameter (UDL Chapter 7) |
| `weights -= r * gradient(...)` | `optimiser.step()` |
| one random pattern per step | a **mini-batch** of patterns per step; one pass through the training set is an **epoch** (UDL Section 6.2) |

First the bridge between the two: one epoch of training in which PyTorch
computes the gradient, but the update is still written out by hand, exactly
as in notebook 1.

In [ ]:
torch.manual_seed(0)
model = nn.Linear(784, 10)
loss_fn = nn.CrossEntropyLoss()
r = 0.1                                       # learning rate
Xtr, ytr = MNIST_DATA['X_train'], MNIST_DATA['y_train']

order = torch.randperm(len(Xtr))
for i in range(0, len(order), 64):            # mini-batches of 64 patterns
    idx = order[i:i + 64]
    loss = loss_fn(model(Xtr[idx]), ytr[idx])
    model.zero_grad()                         # forget the previous gradient
    loss.backward()                           # gradient of the loss, by autograd
    with torch.no_grad():
        for p in model.parameters():          # the update, by hand
            p -= r * p.grad

with torch.no_grad():
    acc = (model(MNIST_DATA['X_test']).argmax(1) == MNIST_DATA['y_test']).float().mean()
print(f'after one epoch: test accuracy {acc:.3f}')

### 2.3 Training and evaluation

`train` below is the same loop, written the way PyTorch code usually is:

* the update is done by an **optimiser** from `torch.optim`, made by
  `make_optimiser`; with `method='sgd'` it is exactly the update above;
* the learning rate may change from epoch to epoch, under a **scheduler**
  made by `make_scheduler`; with `lr_step=0` it stays fixed;
* the data and the model live on the GPU (`device`).

After training, `train` evaluates the model once on the training set and
once on the test set, and records both in the table `RESULTS`.

Only plain SGD with a fixed learning rate is implemented. You add the rest
in Part 3.

In [ ]:
def make_optimiser(model, method, lr, momentum=0.9, weight_decay = 0.):
    """The optimiser for model.parameters()."""
    # TODO (Part 3): pass weight_decay on to the optimiser, for every method.
    # Hint: torch.optim.SGD and torch.optim.Adam take the argument weight_decay=...
    if method == 'sgd':          # stochastic gradient descent (UDL Section 6.2)
        return torch.optim.SGD(model.parameters(), lr=lr)
    if method == 'momentum':     # SGD with momentum (UDL Section 6.3)
        # TODO (Part 3): return SGD with momentum.
        # Hint: torch.optim.SGD takes the argument momentum=...
        raise NotImplementedError("method='momentum' is not implemented yet")
    if method == 'adam':         # Adam (UDL Section 6.4)
        # TODO (Part 3): return the Adam optimiser.
        # Hint: look up torch.optim.Adam.
        raise NotImplementedError("method='adam' is not implemented yet")
    raise ValueError(f'unknown method {method!r}')


def make_scheduler(optimiser, lr_step):
    """A learning-rate schedule: None (fixed rate) if lr_step == 0, otherwise
    the learning rate is halved every lr_step epochs."""
    if lr_step == 0:
        return None
    # TODO (Part 3): halve the learning rate every lr_step epochs.
    # Hint: torch.optim.lr_scheduler.StepLR(optimiser, step_size=..., gamma=...).
    # train() calls scheduler.step() once at the end of every epoch.
    raise NotImplementedError('a learning-rate schedule is not implemented yet')

In [ ]:
RESULTS = []      # one row per run


@torch.no_grad()
def evaluate(model, X, y, bs=1024):
    """Accuracy and mean cross-entropy loss of model on (X, y)."""
    loss_fn = nn.CrossEntropyLoss(reduction='sum')
    total, correct = 0.0, 0
    for i in range(0, len(X), bs):
        scores = model(X[i:i + bs].to(device))
        yb = y[i:i + bs].to(device)
        total += loss_fn(scores, yb).item()
        correct += (scores.argmax(1) == yb).sum().item()
    return correct / len(X), total / len(X)



def train(data, method='sgd', lr=0.1, lr_step=0, epochs=5, batch_size=64, seed=0, weight_decay = 0,
          momentum=0.9):
    """Train logistic regression on data; return the model and its history,
    or None if the method or schedule is not implemented yet."""
    settings = (f"{data['name']}: {method}"
                + (f' {momentum:g}' if method == 'momentum' else '')
                + f', lr {lr:g}'
                + (f', halved every {lr_step} epochs' if lr_step else ', fixed')
                + f', batch {batch_size}, {epochs} epochs'
                + (f', weight_decay {weight_decay}' if weight_decay else ''))
    X_train = data['X_train'].to(device)
    y_train = data['y_train'].to(device)
    torch.manual_seed(seed)
    model = nn.Linear(X_train.shape[1], 10).to(device)   # softmax is in the loss
    try:
        optimiser = make_optimiser(model, method, lr, momentum=momentum, weight_decay=weight_decay)
        scheduler = make_scheduler(optimiser, lr_step)
    except NotImplementedError as e:
        print(f'SKIPPED ({settings}): {e}. Complete the TODO in make_optimiser / make_scheduler.')
        return None
    loss_fn = nn.CrossEntropyLoss()
    history = {'loss': [], 'acc': []}
    t0 = time.time()
    for epoch in range(epochs):
        order = torch.randperm(len(X_train), device=device)
        running, correct = 0.0, 0
        for i in range(0, len(order), batch_size):
            idx = order[i:i + batch_size]
            scores = model(X_train[idx])
            loss = loss_fn(scores, y_train[idx])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            running += loss.item() * len(idx)
            correct += (scores.argmax(1) == y_train[idx]).sum().item()
        if scheduler is not None:
            scheduler.step()
        history['loss'].append(running / len(X_train))
        history['acc'].append(correct / len(X_train))
    tr_acc, tr_loss = evaluate(model, data['X_train'], data['y_train'])
    te_acc, te_loss = evaluate(model, data['X_test'], data['y_test'])
    print(f'{settings}\n  training accuracy {tr_acc:.3f} (loss {tr_loss:.3f})   '
          f'test accuracy {te_acc:.3f} (loss {te_loss:.3f})   [{time.time() - t0:.0f} s]')
    RESULTS.append(dict(settings=settings, train_acc=tr_acc, test_acc=te_acc,
                        train_loss=tr_loss, test_loss=te_loss))
    return model, history

def plot_history(history, title):
    """Training loss and training accuracy per epoch (averaged over the epoch)."""
    ep = range(1, len(history['loss']) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].plot(ep, history['loss']); ax[0].set_title('training loss'); ax[0].set_yscale('log')
    ax[1].plot(ep, history['acc']); ax[1].set_title('training accuracy'); ax[1].set_ylim(0, 1)
    ax[1].axhline(0.1, ls=':', c='gray', label='chance'); ax[1].legend()
    for a in ax: a.set_xlabel('epoch')
    fig.suptitle(title); plt.tight_layout(); plt.show()


def show_templates(model, shape, names):
    """Each row of the weight matrix as an image: one template per class."""
    W = model.weight.detach().cpu()
    fig, axes = plt.subplots(1, 10, figsize=(15, 1.9))
    for i, ax in enumerate(axes):
        w = W[i].view(*shape)
        w = (w - w.min()) / (w.max() - w.min())
        ax.imshow(w.permute(1, 2, 0) if w.dim() == 3 else w, cmap=None if w.dim() == 3 else 'gray')
        ax.set_title(names[i], fontsize=8); ax.axis('off')
    plt.show()

### 2.4 MNIST: the training procedure

Plain SGD with a **fixed learning rate of 0.1**, mini-batches of 64, 5
epochs. This is the **MNIST procedure**; Part 3 starts from it.

In [ ]:
MNIST_PROCEDURE = dict(method='sgd', lr=0.1, lr_step=0, epochs=5, batch_size=64)

mnist_model, hist = train(MNIST_DATA, **MNIST_PROCEDURE)
plot_history(hist, 'MNIST, the MNIST procedure')
show_templates(mnist_model, (28, 28), [str(d) for d in range(10)])

Each class score is the dot product of the image with one row of the weight
matrix, so each row is a **template** for its digit. Bright pixels push the
score of that digit up, dark ones push it down.

## Part 3. Imagenette

### 3.1 Data

**ImageNet** is the collection of photographs on which deep networks for
image recognition made their breakthrough: over a million images in 1,000
classes. Training on it takes days, so **Imagenette** was made as a reduced
version for easier experimentation: about 13,000 photographs of 10 classes —
tench, English springers, cassette players, chain saws, churches, French
horns, garbage trucks, gas pumps, golf balls and parachutes. The official
train and val folders are pooled (13,394 images), shuffled once with a fixed
seed and split 80/20 into a training and a test set. Each image is resized
and centre-cropped to 128×128 pixels, and then treated exactly like an MNIST
image: flattened to one row of $3 \times 128 \times 128 = 49{,}152$ pixel
values in $[0, 1]$.

In [ ]:
CLASS_NAMES = ['tench', 'English springer', 'cassette player', 'chain saw',
               'church', 'French horn', 'garbage truck', 'gas pump',
               'golf ball', 'parachute']
IMG_SIZE = 128


def load_images(root, size):
    """All Imagenette images (train + val folders), resized and centre-cropped
    to size x size, as uint8 tensors. Cached in root/imagenette2_<size>.pt."""
    cache = root / f'imagenette2_{size}.pt'
    if cache.exists():
        d = torch.load(cache)
        return d['X'], d['y']
    if not (root / 'imagenette2').exists():
        datasets.Imagenette(root, split='train', size='full', download=True)
    folders = root / 'imagenette2'
    wnids = sorted(p.name for p in (folders / 'train').iterdir() if p.is_dir())
    tf = transforms.Compose([transforms.Resize(size), transforms.CenterCrop(size),
                             transforms.PILToTensor()])
    files = [(f, c) for split in ('train', 'val') for c, w in enumerate(wnids)
             for f in sorted((folders / split / w).glob('*.JPEG'))]
    X = torch.empty(len(files), 3, size, size, dtype=torch.uint8)
    y = torch.empty(len(files), dtype=torch.long)
    t0 = time.time()
    for i, (f, c) in enumerate(files):
        X[i] = tf(Image.open(f).convert('RGB'))
        y[i] = c
        if (i + 1) % 2000 == 0:
            print(f'  decoded {i + 1}/{len(files)} images ({time.time() - t0:.0f} s)')
    torch.save({'X': X, 'y': y}, cache)
    return X, y


images, labels = load_images(DATA_ROOT, IMG_SIZE)
perm = torch.from_numpy(np.random.default_rng(0).permutation(len(images)))
images, labels = images[perm], labels[perm]
n_train = int(0.8 * len(images))

X = images.reshape(len(images), -1).float() / 255        # 13394 x 49152, in [0, 1]
IMAGENETTE_DATA = dict(name='Imagenette', X_train=X[:n_train], y_train=labels[:n_train],
                       X_test=X[n_train:], y_test=labels[n_train:])
print(f"training: {n_train} images   test: {len(X) - n_train} images   pixels per image: {X.shape[1]}")

fig, axes = plt.subplots(2, 6, figsize=(13, 4.6))
for ax, img, lab in zip(axes.flat, images[:12], labels[:12]):
    ax.imshow(img.permute(1, 2, 0)); ax.set_title(CLASS_NAMES[lab], fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

### 3.2 The MNIST procedure on Imagenette

Exactly the same model and the same training procedure as on MNIST.

In [ ]:
_, hist = train(IMAGENETTE_DATA, **MNIST_PROCEDURE)
plot_history(hist, 'Imagenette, the MNIST procedure')

Look at the training loss and the training accuracy. Has the model learned
the **training set**?

### 3.3 Standardising the inputs

Before changing the optimiser, change the data. **Standardising** each input
means subtracting its mean and dividing by its standard deviation, so that
every input has mean 0 and standard deviation 1. It is a good idea in
general, for any model trained by gradient descent:

* With inputs on very different scales, or all of one sign (pixel values are
  all positive), the loss surface is stretched along some directions. A
  learning rate small enough for the steep directions is then far too small
  for the flat ones, and gradient descent zig-zags or crawls.
* Standardised inputs make the scale of the pre-activations predictable,
  which is what the initialisation schemes of UDL Section 7.5 assume.

The mean and standard deviation are computed on the **training set only**
(here for every pixel position separately) and then applied unchanged to the
test set: the test set plays the part of new data, which we must not look
at while preparing the model.

In [ ]:
mean = IMAGENETTE_DATA['X_train'].mean(dim=0)
std = IMAGENETTE_DATA['X_train'].std(dim=0) + 1e-6       # avoid dividing by 0
IMAGENETTE_STD = dict(name='Imagenette, standardised',
                      X_train=(IMAGENETTE_DATA['X_train'] - mean) / std,
                      y_train=IMAGENETTE_DATA['y_train'],
                      X_test=(IMAGENETTE_DATA['X_test'] - mean) / std,
                      y_test=IMAGENETTE_DATA['y_test'])
del X, images      # free memory
print('training set: mean %.3f, std %.3f' % (IMAGENETTE_STD['X_train'].mean(), IMAGENETTE_STD['X_train'].std()))

### 3.4 Changing the training procedure

Your aim is a training procedure under which logistic regression **succeeds
on the training set** of Imagenette: a training accuracy of at least 0.9.
Things to change, alone or together:

1. **The learning rate and the number of epochs.** A smaller learning rate
   takes smaller steps, so it needs more epochs.
2. **A decreasing learning rate** (`lr_step`): large steps first, smaller
   ones once the loss is near a minimum. Implement `make_scheduler` above.
3. **Momentum** (`method='momentum'`, UDL Section 6.3): the update follows a
   running average of the recent gradients, which smooths out zig-zags.
   Implement it in `make_optimiser`.
4. **Adam** (`method='adam'`, UDL Section 6.4): momentum, plus a step size
   adapted to each parameter separately. Implement it in `make_optimiser`;
   typical learning rates for Adam are 0.001 or smaller.

Each run calls `train(data, method=..., lr=..., lr_step=..., epochs=...,
batch_size=...)`. Put your runs in the list `MY_RUNS` and run the cell; add
runs and run it again as you investigate. Use the standardised data
(`IMAGENETTE_STD`) unless you want to show what standardisation does.

In [ ]:
# TODO: your runs on Imagenette. Each entry is (data, settings).
MY_RUNS = [
    (IMAGENETTE_STD, MNIST_PROCEDURE),     # the MNIST procedure, standardised data
    # (IMAGENETTE_STD, dict(method='sgd', lr=0.001, epochs=20)),
]

for data, settings in MY_RUNS:
    out = train(data, **settings)
    if out is not None:
        plot_history(out[1], f"{data['name']}: {settings}")

The templates of your last successful run (if any) — compare them with the
MNIST templates:

In [ ]:
if 'out' in globals() and out is not None:
    show_templates(out[0], (3, IMG_SIZE, IMG_SIZE), CLASS_NAMES)

## Summary: all your runs

Training and test results of every run in this session. The worksheet asks
you to report them.

In [ ]:
print(f"{'settings':<88}{'train acc':>10}{'test acc':>10}")
for r in RESULTS:
    print(f"{r['settings']:<88}{r['train_acc']:>10.3f}{r['test_acc']:>10.3f}")